# 5. Process South Australia (SA) battery energy storage system (BESS) 44hh Positive-Underlying Datasets

## Purpose
Creates clean SA BESS aggregate datasets by filtering the existing cleaned per-site parquet to the 44 households with no meaningful negative underlying-load intervals.

## Inputs
- `SUMMARY_PATH`

## Run Flow
1. Setup And Path Summary
2. Select The 44 Non-Negative Households
3. Load Selected Per-Site Time Series
4. Aggregate To 30-Minute Mean Power
5. Export PyNNLF Datasets
6. Final Validation Checklist

## Outputs
- `RESULTS_DIR / "sa_bess_44hh_selected_households.csv"`
- `PROCESSED_44_DIR / "sa_bess_44hh_selected_households.csv"`
- `PROCESSED_44_DIR / "sa_bess_44hh_aggregate_5min.parquet"`
- `PROCESSED_44_DIR / "sa_bess_44hh_aggregate_30min_mean_power.csv"`
- `RESULTS_DIR / "sa_bess_44hh_dataset_summary.csv"`
- `PROCESSED_44_DIR / "sa_bess_44hh_pynnlf_dataset_summary.csv"`

## 1. Setup And Path Summary

Inputs stay outside git. Outputs are PyNNLF-ready CSVs written to both root `data/` and publication `data/`.

In [1]:
import os
from pathlib import Path
import sys

def find_publication_project(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "specs").exists() and (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")

def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pynnlf").exists():
            return candidate
    raise FileNotFoundError("Could not find the PyNNLF repo root.")
PROJECT_DIR = find_publication_project()
REPO_ROOT = find_repo_root(PROJECT_DIR)
print(f"Publication project: {PROJECT_DIR}")
print(f"Repository root: {REPO_ROOT}")
import pandas as pd
import pyarrow.dataset as ds
def raw_data_root():
    """Locate the local source data tree, which is not distributed with this repository.

    Set the PYNNLF_RAW_DATA_DIR environment variable to the directory holding the
    "1. raw", "2. processed" and "3. cleaned" folders before running this notebook.

    Returns:
        Path: root of the local source data tree.
    """
    root = os.environ.get("PYNNLF_RAW_DATA_DIR")
    if not root:
        raise RuntimeError(
            "PYNNLF_RAW_DATA_DIR is not set. Point it at your local source data "
            "directory; see the Data section of the repository README."
        )
    return Path(root)


RAW_DATA_ROOT = raw_data_root()

CLEANED_SA_BESS_DIR = RAW_DATA_ROOT / "3. cleaned" / "SA BESS"
SUMMARY_PATH = CLEANED_SA_BESS_DIR / "processed" / "sa_bess_selected_household_summary.csv"
SITE_TIMESERIES_PATH = CLEANED_SA_BESS_DIR / "intermediate" / "sa_bess_selected_site_timeseries_5min.parquet"
PROCESSED_44_DIR = CLEANED_SA_BESS_DIR / "processed" / "44 households positive underlying"
PROCESSED_44_DIR.mkdir(parents=True, exist_ok=True)
ROOT_DATA_DIR = REPO_ROOT / "data"
PUBLICATION_DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results" / "04_sa_bess_clean_44hh"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TARGET_COLUMNS = ["underlying_load_kW", "net_load_with_pv_kW", "net_load_with_pv_and_battery_kW"]
DATASET_SPECS = {
    "ds22": {"filename": "ds22_sa_bess_44hh_pos_underlying_load_30min.csv", "source_column": "underlying_load_kW", "label": "underlying_load"},
    "ds23": {"filename": "ds23_sa_bess_44hh_pos_net_load_with_pv_30min.csv", "source_column": "net_load_with_pv_kW", "label": "net_load_with_pv"},
    "ds24": {"filename": "ds24_sa_bess_44hh_pos_net_load_with_pv_battery_30min.csv", "source_column": "net_load_with_pv_and_battery_kW", "label": "net_load_with_pv_battery"},
}
EXPECTED_HOUSEHOLDS = 44
EXPECTED_5MIN_ROWS_PER_SITE = 105_120
EXPECTED_30MIN_ROWS = 17_520
NEGATIVE_THRESHOLD_KW = -0.05
for path in [SUMMARY_PATH, SITE_TIMESERIES_PATH, ROOT_DATA_DIR, PUBLICATION_DATA_DIR]:
    if not path.exists():
        raise FileNotFoundError(path)
print(f"Summary path: {SUMMARY_PATH}")
print(f"Per-site parquet: {SITE_TIMESERIES_PATH}")
print(f"External 44hh processed folder: {PROCESSED_44_DIR}")

Publication project: <local path redacted>
Repository root: <local path redacted>


Summary path: <local path redacted>
Per-site parquet: <local path redacted>
External 44hh processed folder: <local path redacted>


## 2. Select The 44 Non-Negative Households

The existing diagnostic threshold is used: meaningful negative underlying load means `< -0.05 kW`.

In [2]:
summary = pd.read_csv(SUMMARY_PATH)
summary["site_id"] = summary["site_id"].astype("int64")
summary["selection_rank"] = summary["selection_rank"].astype("int64")
required_summary_columns = {"site_id", "selection_rank", "meaningful_negative_underlying_load_count_post_fill", "negative_load_tolerance_kW", "min_underlying_load_kW_post_fill"}
missing = required_summary_columns - set(summary.columns)
if missing:
    raise ValueError(f"Summary is missing required columns: {sorted(missing)}")
thresholds = sorted(summary["negative_load_tolerance_kW"].dropna().unique().tolist())
if thresholds != [NEGATIVE_THRESHOLD_KW]:
    raise ValueError(f"Expected threshold {NEGATIVE_THRESHOLD_KW}, found {thresholds}")
selected_summary = summary.loc[summary["meaningful_negative_underlying_load_count_post_fill"].eq(0)].sort_values("selection_rank").reset_index(drop=True)
selected_site_ids = selected_summary["site_id"].astype("int64").tolist()
if len(selected_site_ids) != EXPECTED_HOUSEHOLDS:
    raise AssertionError(f"Expected {EXPECTED_HOUSEHOLDS} households, found {len(selected_site_ids)}")
selected_summary.to_csv(RESULTS_DIR / "sa_bess_44hh_selected_households.csv", index=False)
selected_summary.to_csv(PROCESSED_44_DIR / "sa_bess_44hh_selected_households.csv", index=False)
print(f"Selected households: {len(selected_site_ids)}")
display(selected_summary[["selection_rank", "site_id", "state", "postcode", "min_underlying_load_kW_post_fill"]].head(12))

Selected households: 44


,selection_rank,site_id,state,postcode,min_underlying_load_kW_post_fill
0,1,1995273389,SA,5090,0.007010
1,2,245185730,VIC,3453,0.078340
2,3,905921552,NSW,2167,0.148773
3,4,1805446999,VIC,3040,0.168707
4,5,1284666164,VIC,3072,0.209110
5,6,1130090932,NSW,2758,0.033337
6,7,1429376445,NSW,2485,0.267180
7,8,1113741916,TAS,7325,-0.009843
8,9,2140545194,SA,5116,0.020017
9,10,1200712260,SA,5153,0.019017


## 3. Load Selected Per-Site Time Series

The parquet filter avoids loading the excluded 56 households.

In [3]:
site_dataset = ds.dataset(str(SITE_TIMESERIES_PATH), format="parquet")
site_table = site_dataset.to_table(columns=["site_id", "datetime", *TARGET_COLUMNS], filter=ds.field("site_id").isin(selected_site_ids))
site_ts = site_table.to_pandas().sort_values(["site_id", "datetime"]).reset_index(drop=True)
site_ts["site_id"] = site_ts["site_id"].astype("int64")
site_ts["datetime"] = pd.to_datetime(site_ts["datetime"])
expected_rows = EXPECTED_HOUSEHOLDS * EXPECTED_5MIN_ROWS_PER_SITE
if len(site_ts) != expected_rows:
    raise ValueError(f"Expected {expected_rows:,} selected 5-minute rows, found {len(site_ts):,}")
if site_ts[TARGET_COLUMNS].isna().any().any():
    raise ValueError("Selected per-site time series contains missing target values")
if site_ts["underlying_load_kW"].lt(NEGATIVE_THRESHOLD_KW).any():
    bad = int(site_ts["underlying_load_kW"].lt(NEGATIVE_THRESHOLD_KW).sum())
    raise AssertionError(f"Selected cohort still has {bad:,} rows below {NEGATIVE_THRESHOLD_KW} kW")
print(f"Loaded rows: {len(site_ts):,}")
print(f"Date range: {site_ts['datetime'].min()} to {site_ts['datetime'].max()}")
display(site_ts.head())

Loaded rows: 4,625,280
Date range: 2024-02-26 00:00:00 to 2025-02-24 23:55:00


,site_id,datetime,underlying_load_kW,net_load_with_pv_kW,net_load_with_pv_and_battery_kW
0,30236046,2024-02-26 00:00:00,0.543117,0.551133,0.555317
1,30236046,2024-02-26 00:05:00,0.600737,0.608663,0.612760
2,30236046,2024-02-26 00:10:00,0.528720,0.536757,0.540940
3,30236046,2024-02-26 00:15:00,0.490557,0.498663,0.502880
4,30236046,2024-02-26 00:20:00,0.518383,0.526433,0.530593


## 4. Aggregate To 30-Minute Mean Power

The existing SA BESS datasets use arithmetic mean power within each 30-minute window.

In [4]:
aggregate_5min = site_ts.groupby("datetime", as_index=False, observed=True)[TARGET_COLUMNS].sum().sort_values("datetime").reset_index(drop=True)
aggregate_30min = aggregate_5min.set_index("datetime")[TARGET_COLUMNS].resample("30min", label="left", closed="left").mean().reset_index()
if len(aggregate_30min) != EXPECTED_30MIN_ROWS:
    raise ValueError(f"Expected {EXPECTED_30MIN_ROWS:,} 30-minute rows, found {len(aggregate_30min):,}")
if aggregate_30min["datetime"].duplicated().any() or aggregate_30min[TARGET_COLUMNS].isna().any().any():
    raise ValueError("30-minute aggregate contains duplicate timestamps or missing values")
if aggregate_30min["datetime"].diff().dropna().nunique() != 1 or aggregate_30min["datetime"].diff().dropna().iloc[0] != pd.Timedelta(minutes=30):
    raise ValueError("30-minute aggregate is not regular")
aggregate_5min.to_parquet(PROCESSED_44_DIR / "sa_bess_44hh_aggregate_5min.parquet", index=False)
aggregate_30min.to_csv(PROCESSED_44_DIR / "sa_bess_44hh_aggregate_30min_mean_power.csv", index=False)
print(f"5-minute aggregate rows: {len(aggregate_5min):,}")
print(f"30-minute aggregate rows: {len(aggregate_30min):,}")
display(aggregate_30min.head())

5-minute aggregate rows: 105,120
30-minute aggregate rows: 17,520


,datetime,underlying_load_kW,net_load_with_pv_kW,net_load_with_pv_and_battery_kW
0,2024-02-26 00:00:00,31.736383,31.945081,21.285537
1,2024-02-26 00:30:00,30.660720,30.868053,20.859843
2,2024-02-26 01:00:00,28.529513,28.741316,20.851976
3,2024-02-26 01:30:00,29.422615,29.635479,20.921186
4,2024-02-26 02:00:00,28.169503,28.379177,20.635719


## 5. Export PyNNLF Datasets

Each exported CSV contains `datetime,netload_kW`.

In [5]:
dataset_rows = []
for dataset_id, spec in DATASET_SPECS.items():
    dataset = aggregate_30min[["datetime", spec["source_column"]]].rename(columns={spec["source_column"]: "netload_kW"})
    if dataset["netload_kW"].isna().any():
        raise ValueError(f"{dataset_id} contains missing netload_kW values")
    for target_dir in [ROOT_DATA_DIR, PUBLICATION_DATA_DIR]:
        out_path = target_dir / spec["filename"]
        dataset.to_csv(out_path, index=False)
        print(f"Wrote: {out_path}")
    dataset_rows.append({"dataset_id": dataset_id, "filename": spec["filename"], "label": spec["label"], "rows": len(dataset), "start": dataset["datetime"].min(), "end": dataset["datetime"].max(), "min_netload_kW": dataset["netload_kW"].min(), "max_netload_kW": dataset["netload_kW"].max()})
dataset_summary = pd.DataFrame(dataset_rows)
dataset_summary.to_csv(RESULTS_DIR / "sa_bess_44hh_dataset_summary.csv", index=False)
dataset_summary.to_csv(PROCESSED_44_DIR / "sa_bess_44hh_pynnlf_dataset_summary.csv", index=False)
display(dataset_summary)

Wrote: <local path redacted>
Wrote: <local path redacted>
Wrote: <local path redacted>


Wrote: <local path redacted>


Wrote: <local path redacted>
Wrote: <local path redacted>


,dataset_id,filename,label,rows,start,end,min_netload_kW,max_netload_kW
0,ds22,ds22_sa_bess_44hh_pos_underlying_load_30min.csv,underlying_load,17520,2024-02-26,2025-02-24 23:30:00,18.350344,137.519971
1,ds23,ds23_sa_bess_44hh_pos_net_load_with_pv_30min.csv,net_load_with_pv,17520,2024-02-26,2025-02-24 23:30:00,-214.177458,101.366659
2,ds24,ds24_sa_bess_44hh_pos_net_load_with_pv_battery...,net_load_with_pv_battery,17520,2024-02-26,2025-02-24 23:30:00,-199.757353,71.734079


## 6. Final Validation Checklist

In [6]:
for dataset_id, spec in DATASET_SPECS.items():
    path = PUBLICATION_DATA_DIR / spec["filename"]
    df = pd.read_csv(path, parse_dates=["datetime"])
    if list(df.columns) != ["datetime", "netload_kW"]:
        raise ValueError(f"{path.name}: unexpected columns {list(df.columns)}")
    if len(df) != EXPECTED_30MIN_ROWS:
        raise ValueError(f"{path.name}: expected {EXPECTED_30MIN_ROWS:,} rows, found {len(df):,}")
    if df["datetime"].duplicated().any() or df.isna().any().any():
        raise ValueError(f"{path.name}: duplicate timestamps or missing values")
    if df["datetime"].diff().dropna().nunique() != 1 or df["datetime"].diff().dropna().iloc[0] != pd.Timedelta(minutes=30):
        raise ValueError(f"{path.name}: irregular timestamp spacing")
    print(f"{dataset_id}: OK | {path.name} | {df['datetime'].min()} to {df['datetime'].max()} | rows={len(df):,}")

ds22: OK | ds22_sa_bess_44hh_pos_underlying_load_30min.csv | 2024-02-26 00:00:00 to 2025-02-24 23:30:00 | rows=17,520
ds23: OK | ds23_sa_bess_44hh_pos_net_load_with_pv_30min.csv | 2024-02-26 00:00:00 to 2025-02-24 23:30:00 | rows=17,520
ds24: OK | ds24_sa_bess_44hh_pos_net_load_with_pv_battery_30min.csv | 2024-02-26 00:00:00 to 2025-02-24 23:30:00 | rows=17,520
